<span style="color:darkred; font-size:28px; font-weight:bold;">
VERİ KEŞFİ VE TANIMLAYICI İSTATİSTİK
</span>


<div style="text-align: justify;">
📌Bu notebook kapsamında elektrik tüketimi ve tahsilat verileri üzerinde veri keşfi ve tanımlayıcı istatistik analizleri gerçekleştirilmiştir. Analizin amacı veri setinin yapısını anlamak, veri kalitesini değerlendirmek ve ileri aşamalardaki modelleme ve görselleştirme çalışmalarına temel oluşturmaktır. Bu doğrultuda Excel dosyasındaki tüm sayfalar yüklenmiş, veri tipleri ve temel istatistikler incelenmiş, ilçelere göre benzersiz müşteri sayıları karşılaştırılmış ve tahakkuk veri setleri birleştirilerek genel tüketim analizi yapılmıştır. Ayrıca kWh değişkeni için eksik değer, negatif değer ve aşırı uç gözlemler kontrol edilmiş; hesap sınıfına göre ortalama, medyan ve standart sapma tüketim değerleri hesaplanarak veri setinin genel davranışı değerlendirilmiştir.
</div>

<span style="color:#0B3D91; font-size:20px; font-weight:600;">
GEREKLİ KÜTÜPHANELERİN YÜKLENMESİ VE VERİ SETİNİN İÇE AKTARILMASI
</span>

In [1]:
# Gerekli kütüphaneleri import et
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# Excel dosyasını yükle
file_path = 'C:/Users/yaren/Downloads/elektrik_veri_hashed.xlsx'
xls = pd.ExcelFile(file_path)

<span style="color:#0B3D91; font-size:20px; font-weight:600;">
SAYFA İSİMLERİNİN DOĞRULANMASI 
</span>

In [2]:
# Sayfa isimlerini görüntüle
print("Sayfa isimleri:", xls.sheet_names)

Sayfa isimleri: ['Tahsilat', 'Tahsilat 1', 'Tahakkuk', 'Tahakkuk 1', 'Tahakkuk 2']


<span style="color:#0B3D91; font-size:20px; font-weight:600;">
SAYFALARIN AYRI DATAFRAMELERE YÜKLENMESİ
</span

In [3]:
df_tahsilat = pd.read_excel(xls, sheet_name="Tahsilat")
df_tahsilat1 = pd.read_excel(xls, sheet_name="Tahsilat 1")
df_hamamozu = pd.read_excel(xls, sheet_name="Tahakkuk")
df_gumushacikoy = pd.read_excel(xls, sheet_name="Tahakkuk 1")
df_goynucek = pd.read_excel(xls, sheet_name="Tahakkuk 2")

<span style="color:#0B3D91; font-size:20px; font-weight:600;">
HER SAYFANIN BOYUT KONTROLÜ
</span

In [4]:
boyutlar = pd.DataFrame({
    "Satır": [
        df_tahsilat.shape[0],
        df_tahsilat1.shape[0],
        df_hamamozu.shape[0],
        df_gumushacikoy.shape[0],
        df_goynucek.shape[0]
    ],
    "Sütun": [
        df_tahsilat.shape[1],
        df_tahsilat1.shape[1],
        df_hamamozu.shape[1],
        df_gumushacikoy.shape[1],
        df_goynucek.shape[1]
    ]
}, index=[
    "Tahsilat",
    "Tahsilat 1",
    "Hamamözü",
    "Gümüşhacıköy",
    "Göynücek"
])

boyutlar

,Satır,Sütun
Tahsilat,636993,9
Tahsilat 1,917632,22
Hamamözü,124818,10
Gümüşhacıköy,765657,10
Göynücek,295223,10


Boyutların doğruluğu sağlandı.

<span style="color:#0B3D91; font-size:20px; font-weight:600;">
VERİ YAPISI İNCELEME
</span 

In [5]:
dfs = {
    "Tahsilat": df_tahsilat,
    "Tahsilat 1": df_tahsilat1,
    "Hamamözü": df_hamamozu,
    "Gümüşhacıköy": df_gumushacikoy,
    "Göynücek": df_goynucek
}

for isim, df in dfs.items():
    print(f"\n===== {isim} =====")
    
    print("\n--- İlk 5 Satır (.head()) ---")
    display(df.head())
    
    print("\n--- Veri Yapısı (.info()) ---")
    df.info()
    
    print("\n--- İstatistiksel Özet (.describe()) ---")
    display(df.describe())


===== Tahsilat =====

--- İlk 5 Satır (.head()) ---


,Şube,Kasa,İlçe,Söz.hsp.(bağımsız),Tahsilat Tarihi,Nakit Tahsilat,Mahsuben Tahsilat,Kredi Kartı Tahsilatı,Banka Tahsilatı
0,Tayin edilmedi,Tayin edilmedi,TAŞOVA,4989745446,2023-11-06,NaN,8648.95,NaN,NaN
1,Tayin edilmedi,Tayin edilmedi,TAŞOVA,4989745446,2024-06-26,NaN,762.40,NaN,NaN
2,Tayin edilmedi,Tayin edilmedi,TAŞOVA,4989745446,2024-07-10,NaN,311.60,NaN,NaN
3,PTT,PTT/PV,TAŞOVA,4254955886,2023-01-19,NaN,NaN,NaN,130.5
4,PTT,PTT/PV,TAŞOVA,4254955886,2023-02-17,NaN,NaN,NaN,117.0



--- Veri Yapısı (.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 636993 entries, 0 to 636992
Data columns (total 9 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   Şube                   636993 non-null  object        
 1   Kasa                   636993 non-null  object        
 2   İlçe                   636993 non-null  object        
 3   Söz.hsp.(bağımsız)     636993 non-null  int64         
 4   Tahsilat Tarihi        636993 non-null  datetime64[ns]
 5   Nakit Tahsilat         523 non-null     float64       
 6   Mahsuben Tahsilat      7542 non-null    float64       
 7   Kredi Kartı Tahsilatı  0 non-null       float64       
 8   Banka Tahsilatı        628933 non-null  float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(3)
memory usage: 43.7+ MB

--- İstatistiksel Özet (.describe()) ---


,Söz.hsp.(bağımsız),Tahsilat Tarihi,Nakit Tahsilat,Mahsuben Tahsilat,Kredi Kartı Tahsilatı,Banka Tahsilatı
count,6.369930e+05,636993,523.000000,7542.000000,0.0,628933.000000
mean,5.019884e+09,2024-03-05 09:26:07.644856320,694.966635,6180.182282,NaN,372.629109
min,1.758320e+05,2023-01-01 00:00:00,7.450000,-34508.950000,NaN,0.010000
25%,2.532887e+09,2023-07-28 00:00:00,425.330000,44.477500,NaN,120.000000
50%,5.008502e+09,2024-02-26 00:00:00,524.670000,290.410000,NaN,208.000000
75%,7.525722e+09,2024-09-30 00:00:00,688.830000,2729.110000,NaN,322.000000
max,9.999600e+09,2025-05-31 00:00:00,11373.740000,399526.780000,NaN,606473.800000
std,2.884435e+09,NaN,758.319428,23828.022593,NaN,3265.430202



===== Tahsilat 1 =====

--- İlk 5 Satır (.head()) ---


...

,sozlesme_hesap_no,kwh
count,7.656570e+05,765657.000000
mean,5.019916e+09,97.336632
std,2.881724e+09,1077.758336
min,2.561140e+05,-25370.640000
25%,2.519995e+09,18.570000
50%,5.030422e+09,48.310000
75%,7.517065e+09,82.720000
max,9.998331e+09,153575.730000



===== Göynücek =====

--- İlk 5 Satır (.head()) ---


,il,ilce,sozlesme_hesap_no,mali_yil_donem,fatura_tarihi,kayit_tarihi,vade_tarihi,hesap_sinifi,Hesap Sınıfı,kwh
0,AMASYA,GÖYNÜCEK,9374624783,2023-01-01,2023-01-14,2023-03-06,2023-01-24,M001,Mesken,0.10
1,AMASYA,GÖYNÜCEK,9374624783,2023-01-01,2025-03-12,2025-05-09,2025-03-24,M001,Mesken,0.12
2,AMASYA,GÖYNÜCEK,9374624783,2023-02-01,2025-03-12,2025-05-09,2025-03-24,M001,Mesken,0.20
3,AMASYA,GÖYNÜCEK,9374624783,2023-03-01,2025-03-12,2025-05-09,2025-03-24,M001,Mesken,0.00
4,AMASYA,GÖYNÜCEK,9374624783,2023-04-01,2025-03-12,2025-05-09,2025-03-24,M001,Mesken,0.08



--- Veri Yapısı (.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295223 entries, 0 to 295222
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   il                 295223 non-null  object 
 1   ilce               295223 non-null  object 
 2   sozlesme_hesap_no  295223 non-null  int64  
 3   mali_yil_donem     295223 non-null  object 
 4   fatura_tarihi      295223 non-null  object 
 5   kayit_tarihi       295223 non-null  object 
 6   vade_tarihi        295223 non-null  object 
 7   hesap_sinifi       295223 non-null  object 
 8   Hesap Sınıfı       295223 non-null  object 
 9   kwh                295223 non-null  float64
dtypes: float64(1), int64(1), object(8)
memory usage: 22.5+ MB

--- İstatistiksel Özet (.describe()) ---


,sozlesme_hesap_no,kwh
count,2.952230e+05,295223.000000
mean,4.950566e+09,89.669891
std,2.887724e+09,742.276369
min,1.118550e+06,-4208.640000
25%,2.432854e+09,17.860000
50%,4.928206e+09,45.090000
75%,7.475616e+09,77.140000
max,9.997185e+09,105687.690000


In [6]:
def tip_ozeti(df):
    return pd.DataFrame({
        "Veri Tipi": df.dtypes,
        "Eksik Değer": df.isna().sum()
    })

for isim, df in dfs.items():
    print(f"\n===== {isim} =====")
    display(tip_ozeti(df))


===== Tahsilat =====


,Veri Tipi,Eksik Değer
Şube,object,0
Kasa,object,0
İlçe,object,0
Söz.hsp.(bağımsız),int64,0
Tahsilat Tarihi,datetime64[ns],0
Nakit Tahsilat,float64,636470
Mahsuben Tahsilat,float64,629451
Kredi Kartı Tahsilatı,float64,636993
Banka Tahsilatı,float64,8060



===== Tahsilat 1 =====


,Veri Tipi,Eksik Değer
Mali yıl/dönem,object,0
İl,object,0
İlçe,object,0
Söz.hsp.(bağımsız),int64,0
Hesap Sınıfı,object,0
Tahakkuk Tutar,float64,0
Son Ödeme Tarihinden Önceki Tahsilat,float64,293724
Son Ödeme Tarihindeki Tahsilat,float64,589439
Son Ödeme (1),float64,896730
Son Ödeme (2),float64,895968



===== Hamamözü =====


,Veri Tipi,Eksik Değer
il,object,0
ilce,object,0
sozlesme_hesap_no,int64,0
mali_yil_donem,object,0
fatura_tarihi,object,0
kayit_tarihi,object,0
vade_tarihi,object,0
hesap_sinifi,object,0
Hesap Sınıfı,object,0
kwh,float64,0



===== Gümüşhacıköy =====


,Veri Tipi,Eksik Değer
il,object,0
ilce,object,0
sozlesme_hesap_no,int64,0
mali_yil_donem,object,0
fatura_tarihi,object,0
kayit_tarihi,object,0
vade_tarihi,object,0
hesap_sinifi,object,0
Hesap Sınıfı,object,0
kwh,float64,0



===== Göynücek =====


,Veri Tipi,Eksik Değer
il,object,0
ilce,object,0
sozlesme_hesap_no,int64,0
mali_yil_donem,object,0
fatura_tarihi,object,0
kayit_tarihi,object,0
vade_tarihi,object,0
hesap_sinifi,object,0
Hesap Sınıfı,object,0
kwh,float64,0


İlçe dataframelerinde bulunan tarih değişkenleri bir sonraki notebookta mevimsellik analizleri için datatime formatına çevrilecektir.

<span style="color:#0B3D91; font-size:20px; font-weight:600;">
HER İLÇE İÇİN BENZERSİZ MÜŞTERİ SAYISI
</span

In [7]:
# Benzersiz müşteri sayısı
musteri_sayilari = {
    "Hamamözü": df_hamamozu['sozlesme_hesap_no'].nunique(),
    "Gümüşhacıköy": df_gumushacikoy['sozlesme_hesap_no'].nunique(),
    "Göynücek": df_goynucek['sozlesme_hesap_no'].nunique()
}

musteri_df = pd.DataFrame.from_dict(musteri_sayilari, orient='index', columns=['Benzersiz Müşteri Sayısı'])
musteri_df

,Benzersiz Müşteri Sayısı
Hamamözü,2981
Gümüşhacıköy,18190
Göynücek,7128


Veri bilgi dokümanında belirtilen kayıt sayıları ile elde edilen sonuçların uyumlu olduğu doğrulanmıştır. Ayrıca ilçelere göre benzersiz müşteri sayıları incelendiğinde Gümüşhacıköy ilçesinin 18190 müşteri ile diğer ilçelere kıyasla belirgin şekilde daha yüksek bir müşteri tabanına sahip olduğu görülmektedir. Göynücek ilçesi 7128 müşteri ile orta seviyede yer alırken, Hamamözü ilçesi 2981 müşteri ile en düşük müşteri sayısına sahiptir. Bu durum, ilçeler arasında nüfus yoğunluğu, ekonomik faaliyetler veya elektrik abonelik yapısı açısından farklılıklar olabileceğine işaret etmektedir.

<span style="color:#0B3D91; font-size:20px; font-weight:600;">
TÜM TAHAKKUK VERİLERİNİ BİRLEŞTİRME
</span

In [8]:
# Birleştirme
tahakkuk_all = pd.concat([df_hamamozu, df_gumushacikoy, df_goynucek], ignore_index=True)

#Kayıt sayısını doğrulayalım
print("Toplam birleşik kayıt:", len(tahakkuk_all))

print("Beklenen:",
      len(df_hamamozu) +
      len(df_gumushacikoy) +
      len(df_goynucek))

Toplam birleşik kayıt: 1185698
Beklenen: 1185698


In [9]:
tahakkuk_all["ilce"].value_counts()

ilce
GÜMÜŞHACIKÖY    765657
GÖYNÜCEK        295223
HAMAMÖZÜ        124818
Name: count, dtype: int64

İlçe bazlı dağılım incelenmiş, kayıtların doğru şekilde birleştiği doğrulanmıştır.

<span style="color:#0B3D91; font-size:20px; font-weight:600;">
KWH SÜTUNUNDAKİ EKSİK,NEGATİF VE AŞIRI UÇ DEĞERLERİ TESPİT ETME
</span

In [10]:
# --- 1. KWH VERİ KALİTE KONTROLÜ ---

# A. Eksik Değerler
eksik_sayisi = tahakkuk_all['kwh'].isna().sum()
eksik_orani = (eksik_sayisi / len(tahakkuk_all)) * 100

# B. Negatif Değerler (Fiziksel olarak mümkün olmayan durumlar)
negatif_sayisi = (tahakkuk_all['kwh'] < 0).sum()
negatif_veriler = tahakkuk_all[tahakkuk_all['kwh'] < 0]

# C. Aşırı Uç Değerler (IQR Yöntemi ile)
Q1 = tahakkuk_all['kwh'].quantile(0.25)
Q3 = tahakkuk_all['kwh'].quantile(0.75)
IQR = Q3 - Q1

alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

outlier_df = tahakkuk_all[(tahakkuk_all['kwh'] < alt_sinir) | (tahakkuk_all['kwh'] > ust_sinir)]
outlier_sayisi = len(outlier_df)

# --- 2. SONUÇLARI RAPORLAMA ---
print("="*40)
print("🔍 KWH SÜTUNU KALİTE RAPORU")
print("="*40)
print(f"Eksik Değer Sayısı: {eksik_sayisi} (%{eksik_orani:.4f})")
print(f"Negatif Değer Sayısı: {negatif_sayisi}")
print(f"Aşırı Uç (Outlier) Sayısı: {outlier_sayisi}")
print(f"Normal Değer Aralığı: {alt_sinir:.2f} ile {ust_sinir:.2f} arası")
print("-" * 40)

# Outlier'ların hangi hesap sınıflarında olduğunu görmek çok kritiktir!
print("\n>>> En Çok Outlier Görülen Hesap Sınıfları (Top 5):")
display(outlier_df['Hesap Sınıfı'].value_counts().head(5))

🔍 KWH SÜTUNU KALİTE RAPORU
Eksik Değer Sayısı: 0 (%0.0000)
Negatif Değer Sayısı: 151
Aşırı Uç (Outlier) Sayısı: 48554
Normal Değer Aralığı: -74.97 ile 172.98 arası
----------------------------------------

>>> En Çok Outlier Görülen Hesap Sınıfları (Top 5):


Hesap Sınıfı
Mesken                                     20922
Ticari Faaliyet - Yazıhane                 15818
1 SAYILI CETVELDE YER ALAN KAMU İDARESİ     2260
Tarımsal Faaliyetler (Şahıs)                1896
Köy İçme Suyu Temini ve Dağıtımı Tesisi     1315
Name: count, dtype: int64

In [11]:
negatif_df = tahakkuk_all[tahakkuk_all['kwh'] < 0].copy()
negatif_ozet = (
    negatif_df.groupby("Hesap Sınıfı")
    .size()
    .reset_index(name='Negatif Kayıt Sayısı')
    .sort_values(by='Negatif Kayıt Sayısı', ascending=False)
    .head(10) # En çok negatif olan ilk 10'u alalım
)


# Sayısal tabloyu göster
print("--- Negatif Değer İçeren Hesap Sınıfları Detaylı Listesi ---")
display(negatif_ozet)

--- Negatif Değer İçeren Hesap Sınıfları Detaylı Listesi ---


,Hesap Sınıfı,Negatif Kayıt Sayısı
1,Mesken,106
4,Tarımsal Faaliyetler (Kooperatif),16
6,Ticari Faaliyet - Yazıhane,13
5,Tarımsal Faaliyetler (Şahıs),10
2,Resmi Daire,2
7,Şantiye ve Geçici Aboneler,2
0,Belediye,1
3,Süt Toplama Merkezi,1


In [12]:
tahakkuk_clean = tahakkuk_all[tahakkuk_all['kwh'] >= 0]

Negatif kwh değerleri toplam veriye oranla çok az olsa bile fiziksel olarak bu durum mümkün olmadığından sonraki analizler için veri setinden silinmiştir.

In [13]:
ilce_stats = tahakkuk_clean.groupby('ilce')['kwh'].agg(
    mean='mean', median='median', std='std', min='min', max='max',
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75),
    outlier_count=lambda x: ((x < alt_sinir) | (x > ust_sinir)).sum()
).sort_values(by='mean', ascending=False)

ilce_stats['outlier_ratio_%'] = ilce_stats['outlier_count'] / tahakkuk_clean.groupby('ilce')['kwh'].count() * 100
ilce_stats

,mean,median,std,min,max,Q1,Q3,outlier_count,outlier_ratio_%
ilce,,,,,,,,,
GÜMÜŞHACIKÖY,97.473146,48.32,1077.223636,0.0,153575.73,18.59,82.73,31757,4.148259
GÖYNÜCEK,89.736998,45.10,742.244714,0.0,105687.69,17.87,77.14,12337,4.179441
HAMAMÖZÜ,70.892741,40.56,389.199585,0.0,25941.60,15.49,70.43,4356,3.489993


<span style="color:#0B3D91; font-size:20px; font-weight:600;">
HESAP SINIFINA GÖRE İSTATİSTİK TABLOSU
</span

In [14]:
hesap_stats = (
   tahakkuk_clean
    .groupby("Hesap Sınıfı")["kwh"]
    .agg(["count","mean","median","std"])
    .sort_values("mean", ascending=False)
)

hesap_stats

,count,mean,median,std
Hesap Sınıfı,,,,
Karayolları Genel Müdürlüğü Aydınlatma,41,30203.431220,36810.900,16963.572539
Aritma Tesisleri,35,16594.174857,16186.910,11656.705924
Lisansız Üreticiler,180,16155.254444,322.320,35926.424902
Sanayi,187,7293.799626,1531.680,13716.343807
İçme-Kullanma Suyu (Belediye),417,5213.509281,400.020,8948.065839
Tarımsal Faaliyetler (Kooperatif),1616,3830.021999,282.720,9454.764009
Lisansız Üreticiler - Resmi Daire,36,1195.051389,770.625,1229.579483
"Resmi SAĞLIK KURULUŞLARI,RESMİ SPOR TES.",442,1154.739118,239.905,3804.865088
"Resmi Üniversite,Yük.Okul,Kurs,Yurt,Okul",476,883.888739,285.410,1435.311536


In [15]:
# Tahakkuk clean verisi
tahakkuk_clean.to_pickle("tahakkuk_clean.pkl")

# Tahsilat verileri
df_tahsilat.to_pickle("tahsilat_clean.pkl")
df_tahsilat1.to_pickle("tahsilat1_clean.pkl")

Dataframeler üzerinde değişken tipi değişimi, negatif veri silme işlemlerini yaptım ve bunları görselleştirme ve veri hikayesi aşamalarında da kullanmak istediğim için kaydettim.İlk olarak parquetle kaydetmeyi denedim çalışmadığı için başka bir alternatif olan pickle ile kaydetmeyi denedim çalıştı. Ayrıca dataframelerin okunması yaklaşık 7-8 dakika sürdüğü içinde bu şekilde kaydetme şekline başvurdum.